In [ ]:
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.ensemble import AdaBoostClassifier, BaggingClassifier, RandomForestClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, RandomizedSearchCV
from catboost import CatBoostClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score, roc_curve, accuracy_score, precision_score, recall_score
import time
import shap
from sklearn.decomposition import PCA
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

In [105]:
class OutlierHandler(BaseEstimator, TransformerMixin):
    def __init__(self, factor=1.5):
        self.factor = factor
        self.lower_bounds = {}
        self.upper_bounds = {}

    def fit(self, X, y=None):
        X_df = pd.DataFrame(X)
        for col in X_df.columns:
            Q1 = X_df[col].quantile(0.25)
            Q3 = X_df[col].quantile(0.75)
            IQR = Q3 - Q1
            self.lower_bounds[col] = Q1 - self.factor * IQR
            self.upper_bounds[col] = Q3 + self.factor * IQR
        return self

    def transform(self, X, y=None):
        X_df = pd.DataFrame(X).copy()
        for col in X_df.columns:
            # Capping (Winsorization)
            X_df[col] = np.where(X_df[col] < self.lower_bounds[col], self.lower_bounds[col], X_df[col])
            X_df[col] = np.where(X_df[col] > self.upper_bounds[col], self.upper_bounds[col], X_df[col])
        return X_df.values

In [106]:
df = pd.read_csv('./dataset/Kamil_kg_V/heart_preprocess.csv')

In [107]:
df.duplicated().sum()

np.int64(0)

In [108]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 301717 entries, 0 to 301716
Data columns (total 18 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   Unnamed: 0        301717 non-null  int64  
 1   HeartDisease      301717 non-null  int64  
 2   BMI               301717 non-null  float64
 3   Smoking           301717 non-null  int64  
 4   AlcoholDrinking   301717 non-null  int64  
 5   Stroke            301717 non-null  int64  
 6   PhysicalHealth    301717 non-null  float64
 7   MentalHealth      301717 non-null  float64
 8   DiffWalking       301717 non-null  int64  
 9   Sex               301717 non-null  int64  
 10  AgeCategory       278365 non-null  float64
 11  Diabetic          292385 non-null  float64
 12  PhysicalActivity  301717 non-null  int64  
 13  GenHealth         301717 non-null  int64  
 14  SleepTime         301717 non-null  float64
 15  Asthma            301717 non-null  int64  
 16  KidneyDisease     30

In [109]:
df['HeartDisease'].value_counts()

HeartDisease
0    274456
1     27261
Name: count, dtype: int64

In [ ]:
X = df.drop('HeartDisease', axis=1)
y = df['HeartDisease']

X_train,X_test,Y_train,Y_test = train_test_split(X,y,test_size=0.2,stratify=y)


list_col_full = [
    'age','sex','cp','trestbps','chol','fbs','restecg','thalach','exang','oldpeak','slope','ca','thal'
]

num_features = ['BMI', 'PhysicalHealth', 'MentalHealth', 'SleepTime']
cat_features = ["Smoking",
    "AlcoholDrinking",
    "Stroke",
    "DiffWalking",
    "Diabetic",
    "PhysicalActivity",
    "Asthma",
    "KidneyDisease",
    "SkinCancer",
    "Sex"]

In [ ]:
# Tính tỷ lệ lớp đa số / lớp thiểu số để dùng cho XGBoost
neg_count = (Y_train == 0).sum()
pos_count = (Y_train == 1).sum()
scale_pos_weight_val = neg_count / pos_count

print(f"Tỷ lệ scale_pos_weight cho XGBoost: {scale_pos_weight_val:.2f}")

In [ ]:
print(f"Train set size: {len(X_train)} (Positive: {(Y_train==1).sum()}, Negative: {(Y_train==0).sum()})")
print(f"Test set size: {len(X_test)} (Positive: {(Y_test==1).sum()}, Negative: {(Y_test==0).sum()})")
print(f"Tỷ lệ positive trong train: {(Y_train==1).mean():.2%}")
print(f"Tỷ lệ positive trong test: {(Y_test==1).mean():.2%}")

PREPROCESSING DATA KNN-SVM-LR

In [111]:
num_pipeline_scaled = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("outlier", OutlierHandler()),
    ("scaler", StandardScaler())
])

cat_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocess_scaled = ColumnTransformer([
    ("num", num_pipeline_scaled, num_features),
    ("cat", cat_pipeline, cat_features)
])

Decision Tree, Random Forest, Bagging

In [112]:
num_pipeline_no_scale = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

preprocess_tree = ColumnTransformer([
    ("num", num_pipeline_no_scale, num_features),
    ("cat", cat_pipeline, cat_features)
])

XGBoost, AdaBoost

In [113]:
preprocess_boost = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median"))
    ]), num_features),
    
    ("cat", cat_pipeline, cat_features)
])

In [114]:
CatBoostClassifier(cat_features=cat_features)

CatBoostClassifier(cat_features=['Smoking', 'AlcoholDrinking', 'Stroke', 'DiffWalking', 'Diabetic', 'PhysicalActivity', 'Asthma', 'KidneyDisease', 'SkinCancer', 'Sex'])

In [ ]:
pipe_knn = Pipeline([
    ("preprocess", preprocess_scaled),
    ("model", KNeighborsClassifier())
])

pipe_lr = Pipeline([
    ("preprocess", preprocess_scaled),
    ("model", LogisticRegression(max_iter=1000, class_weight="balanced"))
])

pipe_svm = Pipeline([
    ("preprocess", preprocess_scaled),
    ("model", SVC(probability=True, class_weight="balanced"))
])

pipe_dt = Pipeline([
    ("preprocess", preprocess_tree),
    ("model", DecisionTreeClassifier(class_weight="balanced"))
])

pipe_rf = Pipeline([
    ("preprocess", preprocess_tree),
    ("model", RandomForestClassifier(class_weight="balanced"))
])

pipe_bagging = Pipeline([
    ("preprocess", preprocess_tree),
    ("model", BaggingClassifier(
        estimator=DecisionTreeClassifier(class_weight="balanced"),
        n_estimators=100
    ))
])

pipe_xgb = Pipeline([
    ("preprocess", preprocess_boost),
    ("model", XGBClassifier(eval_metric='logloss', scale_pos_weight=scale_pos_weight_val))
])

pipe_ada = Pipeline([
    ("preprocess", preprocess_boost),
    ("model", AdaBoostClassifier())
])

pipe_cat = Pipeline([
    ("preprocess", preprocess_boost),
    ("model", CatBoostClassifier(verbose=0, auto_class_weights='Balanced'))
])

In [116]:
pipelines = {
    "KNN": pipe_knn,
    "LR": pipe_lr,
    "SVM": pipe_svm,
    "DT": pipe_dt,
    "RF": pipe_rf,
    "Bagging": pipe_bagging,
    "XGB": pipe_xgb,
    "Ada": pipe_ada,
    "CatBoost": pipe_cat
}

In [ ]:
param_knn = {
    "model__n_neighbors": [5,7,11],
    "model__weights": ["distance"],
    "model__p": [2]
}

param_lr = {
    "model__C": [0.01, 0.1, 1, 10, 100],
    "model__penalty": ["l2"],
    "model__solver": ["lbfgs"]
}

param_svm = {
    "model__C": [0.1, 1, 10],
    "model__kernel": ["rbf"],
    "model__gamma": ["scale"]
}

param_dt = {
    "model__max_depth": [3,5,10,20,None],
    "model__min_samples_split": [2,5,10],
    "model__min_samples_leaf": [1,2,4],
    "model__criterion": ["gini", "entropy"]
}

param_rf = {
    "model__n_estimators": [100,200,300],
    "model__max_depth": [None,5,10,20],
    "model__min_samples_split": [2,5],
    "model__min_samples_leaf": [1,2],
    "model__max_features": ["sqrt", "log2"]
}

param_bagging = {
    "model__n_estimators": [50,100,200],
    "model__max_samples": [0.5, 0.7, 1.0],
    "model__max_features": [0.5, 0.7, 1.0]
}

param_xgb = {
    "model__n_estimators": [100,200,300],
    "model__max_depth": [3,5,7],
    "model__learning_rate": [0.01, 0.1, 0.2],
    "model__subsample": [0.8, 1.0],
    "model__colsample_bytree": [0.8, 1.0]
}

param_ada = {
    "model__n_estimators": [50,100,200],
    "model__learning_rate": [0.01, 0.1, 1]
}

param_cat = {
    "model__iterations": [100,200,300],
    "model__depth": [4,6,8],
    "model__learning_rate": [0.01, 0.1],
    "model__l2_leaf_reg": [1,3,5]
}

In [118]:
param_grids = {
    "KNN": param_knn,
    "LR": param_lr,
    "SVM": param_svm,
    "DT": param_dt,
    "RF": param_rf,
    "Bagging": param_bagging,
    "XGB": param_xgb,
    "Ada": param_ada,
    "CatBoost": param_cat
}

ĐÁNH GIÁ BASELINE

In [119]:
# baseline_results = []
# for name, pipe in pipelines.items():
#     # Cross-validation score với default params
#     cv_scores = cross_val_score(pipe, X_train, Y_train, cv=5, scoring='roc_auc')
#     baseline_results.append({
#         "Model": name,
#         "CV AUC Mean": cv_scores.mean(),
#         "CV AUC Std": cv_scores.std()
#     })

# baseline_df = pd.DataFrame(baseline_results).sort_values("CV AUC Mean", ascending=False)
# print(baseline_df.to_string(index=False))

In [ ]:
# Stratified K-Fold để giữ tỷ lệ classes trong mỗi fold
stratified_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# giữ nguyên tất cả positive, lấy đủ negative
X_train_balanced, _, y_train_balanced, _ = train_test_split(
    X_train, Y_train,
    train_size=30000,
    stratify=Y_train,   # Giữ nguyên tỷ lệ gốc
    random_state=42
)
print(f"Training subset: {len(X_train_balanced)} samples")
print(f"Positive samples: {(y_train_balanced==1).sum()} ({(y_train_balanced==1).mean():.2%})")

In [ ]:
best_models = {}
evaluation_results = []

for name, pipe in pipelines.items():
    print(f"[{name}] Đang huấn luyện với GridSearchCV...")
    
    # 1. Huấn luyện GridSearchCV
    grid = RandomizedSearchCV(
        pipe,
        param_grids[name],
        cv=stratified_cv,
        n_iter=10,
        scoring="roc_auc",
        n_jobs=-1,
        return_train_score=True,
        verbose=1
    )
    
    start_time = time.time()
    
    grid.fit(X_train_balanced, y_train_balanced)
    
    end_time = time.time()
    training_time = end_time - start_time
    
    best_models[name] = grid.best_estimator_

    
    # 2. Dự đoán xác suất cho nhãn Positive (1)
    # SVM của bạn đã có probability=True nên predict_proba sẽ chạy bình thường
    y_probs = grid.best_estimator_.predict_proba(X_test)[:, 1]
    
    # 3. Tính toán ROC AUC và tìm ngưỡng cắt (Threshold) tối ưu
    auc = roc_auc_score(Y_test, y_probs)
    fpr, tpr, thresholds = roc_curve(Y_test, y_probs)
    
    # Sử dụng Youden's J statistic để tìm threshold cắt tối ưu (tối đa hoá tpr - fpr)
    J = tpr - fpr
    optimal_idx = np.argmax(J)
    optimal_threshold = thresholds[optimal_idx]
    
    # 4. Dự đoán dựa trên ngưỡng vừa tìm được
    y_pred = (y_probs >= optimal_threshold).astype(int)
    
    # 5. Đánh giá các độ đo khác
    acc = accuracy_score(Y_test, y_pred)
    prec = precision_score(Y_test, y_pred, zero_division=0)
    rec = recall_score(Y_test, y_pred, zero_division=0)
    
    evaluation_results.append({
        "Model": name,
        "Training Time (s)": round(training_time, 4),
        "AUC": auc,
        "Optimal Threshold": optimal_threshold,
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "Best Params": grid.best_params_
    })
    print(f"  ✓ AUC: {auc:.4f}, F1: {f1:.4f}, Time: {training_time:.1f}s")

[KNN] Đang huấn luyện với GridSearchCV...
Fitting 5 folds for each of 10 candidates, totalling 50 fits
[LR] Đang huấn luyện với GridSearchCV...


c:\Users\admin\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\model_selection\_search.py:324: UserWarning: The total space of parameters 5 is smaller than n_iter=10. Running 5 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


Fitting 5 folds for each of 5 candidates, totalling 25 fits


c:\Users\admin\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


[SVM] Đang huấn luyện với GridSearchCV...
Fitting 5 folds for each of 10 candidates, totalling 50 fits


KeyboardInterrupt: 

In [ ]:
results_df = pd.DataFrame(evaluation_results)
results_df = results_df.sort_values("AUC", ascending=False).reset_index(drop=True)

pd.set_option('display.max_colwidth', 50)
print(results_df[["Model", "AUC", "F1-Score", "Precision", "Recall", "Train Time (s)"]].to_string(index=False))

In [ ]:
pca_step = PCA()

pipe_knn = Pipeline([
    ("preprocess", preprocess_scaled),
    ("pca", pca_step),
    ("model", KNeighborsClassifier())
])

pipe_lr = Pipeline([
    ("preprocess", preprocess_scaled),
    ("pca", pca_step),
    ("model", LogisticRegression(max_iter=1000, class_weight="balanced"))
])

pipe_svm = Pipeline([
    ("preprocess", preprocess_scaled),
    ("pca", pca_step),
    ("model", SVC(probability=True, class_weight="balanced"))
])

pipe_dt = Pipeline([
    ("preprocess", preprocess_tree),
    ("pca", pca_step),
    ("model", DecisionTreeClassifier(class_weight="balanced"))
])

pipe_rf = Pipeline([
    ("preprocess", preprocess_tree),
    ("pca", pca_step),
    ("model", RandomForestClassifier(class_weight="balanced"))
])

pipe_bagging = Pipeline([
    ("preprocess", preprocess_tree),
    ("pca", pca_step),
    ("model", BaggingClassifier(
        estimator=DecisionTreeClassifier(class_weight="balanced"),
        n_estimators=100
    ))
])

pipe_xgb = Pipeline([
    ("preprocess", preprocess_boost),
    ("pca", pca_step),
    ("model", XGBClassifier(eval_metric='logloss', scale_pos_weight=scale_pos_weight_val))
])

pipe_ada = Pipeline([
    ("preprocess", preprocess_boost),
    ("pca", pca_step),
    ("model", AdaBoostClassifier())
])

pipe_cat = Pipeline([
    ("preprocess", preprocess_boost),
    ("pca", pca_step),
    ("model", CatBoostClassifier(verbose=0, auto_class_weights='Balanced'))
])

In [ ]:
param_knn = {
    "pca__n_components": [0.85, 0.90, 0.95],
    "model__n_neighbors": [3, 5, 7, 9, 11],
    "model__weights": ["uniform", "distance"],
    "model__p": [1, 2]
}

param_lr = {
    "pca__n_components": [0.85, 0.90, 0.95],
    "model__C": [0.01, 0.1, 1, 10, 100],
    "model__penalty": ["l2"],
    "model__solver": ["lbfgs"]
}

param_svm = {
    "pca__n_components": [0.85, 0.90, 0.95],
    "model__C": [0.1, 1, 10, 100],
    "model__kernel": ["rbf", "linear"],
    "model__gamma": ["scale", "auto"]
}

param_dt = {
    "pca__n_components": [0.85, 0.90, 0.95],
    "model__max_depth": [3, 5, 10, 20, None],
    "model__min_samples_split": [2, 5, 10],
    "model__min_samples_leaf": [1, 2, 4],
    "model__criterion": ["gini", "entropy"]
}

param_rf = {
    "pca__n_components": [0.85, 0.90, 0.95],
    "model__n_estimators": [100, 200, 300],
    "model__max_depth": [None, 5, 10, 20],
    "model__min_samples_split": [2, 5],
    "model__min_samples_leaf": [1, 2]
}

param_bagging = {
    "pca__n_components": [0.85, 0.90, 0.95],
    "model__n_estimators": [50, 100, 200],
    "model__max_samples": [0.5, 0.7, 1.0],
    "model__max_features": [0.5, 0.7, 1.0]
}

param_xgb = {
    "pca__n_components": [0.85, 0.90, 0.95],
    "model__n_estimators": [100, 200, 300],
    "model__max_depth": [3, 5, 7],
    "model__learning_rate": [0.01, 0.1, 0.2],
    "model__subsample": [0.8, 1.0],
    "model__colsample_bytree": [0.8, 1.0]
}

param_ada = {
    "pca__n_components": [0.85, 0.90, 0.95],
    "model__n_estimators": [50, 100, 200],
    "model__learning_rate": [0.01, 0.1, 1]
}

param_cat = {
    "pca__n_components": [0.85, 0.90, 0.95],
    "model__iterations": [100, 200, 300],
    "model__depth": [4, 6, 8],
    "model__learning_rate": [0.01, 0.1]
    # Bỏ l2_leaf_reg đi bớt để giảm thời gian train nếu bạn kết hợp cả PCA
}

param_grids = {
    "KNN": param_knn,
    "LR": param_lr,
    "SVM": param_svm,
    "DT": param_dt,
    "RF": param_rf,
    "Bagging": param_bagging,
    "XGB": param_xgb,
    "Ada": param_ada,
    "CatBoost": param_cat
}

In [ ]:
best_models = {}
evaluation_results = []

for name, pipe in pipelines.items():
    print(f"[{name}] Đang huấn luyện với GridSearchCV...")
    
    # 1. Huấn luyện GridSearchCV
    grid = RandomizedSearchCV(
        pipe,
        param_grids[name],
        cv=stratified_cv,
        n_iter=10,
        scoring="neg_log_loss",
        n_jobs=-1,
        return_train_score=True,
        verbose=1
    )
    
    start_time = time.time()
    
    grid.fit(X_train_balanced, y_train_balanced)
    
    end_time = time.time()
    training_time = end_time - start_time
    
    best_models[name] = grid.best_estimator_
    
    
    best_idx = grid.best_index_
    
    # neg_log_loss trả về số âm, nên ta nhân -1 để có giá trị log_loss chuẩn (càng nhỏ càng tốt)
    train_loss = -grid.cv_results_['mean_train_score'][best_idx]
    val_loss_cv = -grid.cv_results_['mean_test_score'][best_idx]
    
    # 2. Dự đoán xác suất cho nhãn Positive (1)
    # SVM của bạn đã có probability=True nên predict_proba sẽ chạy bình thường
    y_probs = grid.best_estimator_.predict_proba(X_test)[:, 1]
    
    # 3. Tính toán ROC AUC và tìm ngưỡng cắt (Threshold) tối ưu
    auc = roc_auc_score(Y_test, y_probs)
    fpr, tpr, thresholds = roc_curve(Y_test, y_probs)
    
    # Sử dụng Youden's J statistic để tìm threshold cắt tối ưu (tối đa hoá tpr - fpr)
    J = tpr - fpr
    optimal_idx = np.argmax(J)
    optimal_threshold = thresholds[optimal_idx]
    
    # 4. Dự đoán dựa trên ngưỡng vừa tìm được
    y_pred = (y_probs >= optimal_threshold).astype(int)
    
    # 5. Đánh giá các độ đo khác
    acc = accuracy_score(Y_test, y_pred)
    prec = precision_score(Y_test, y_pred, zero_division=0)
    rec = recall_score(Y_test, y_pred, zero_division=0)
    
    evaluation_results.append({
        "Model": name,
        "Training Time (s)": round(training_time, 4),
        "AUC": auc,
        "Optimal Threshold": optimal_threshold,
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "Best Params": grid.best_params_
    })

In [ ]:
results_df = pd.DataFrame(evaluation_results)
results_df = results_df.sort_values("AUC", ascending=False).reset_index(drop=True)

pd.set_option('display.max_colwidth', 50)
print(results_df[["Model", "AUC", "F1-Score", "Precision", "Recall", "Train Time (s)"]].to_string(index=False))

In [ ]:
# # ============================================
# # 11. KIỂM TRA TIỀN XỬ LÝ
# # ============================================
# print("\n" + "="*60)
# print("PREPROCESSING VALIDATION")
# print("="*60)

# # Kiểm tra missing values
# print("\n1. Missing values after preprocessing:")
# for name, pipe in pipelines.items():
#     # Transform a small sample
#     try:
#         X_transformed = pipe.named_steps['preprocess'].fit_transform(X_train.head(5))
#         print(f"   {name}: {X_transformed.shape} - No missing values (handled by SimpleImputer)")
#     except Exception as e:
#         print(f"   {name}: Error - {e}")

# # Kiểm tra outlier handling
# print("\n2. Outlier handling check (on 'age' feature):")
# age_before = X_train['age'].values
# # Lấy age sau khi qua outlier handler
# age_pipeline = Pipeline([
#     ("imputer", SimpleImputer(strategy="median")),
#     ("outlier", OutlierHandler(factor=1.5))
# ])
# age_after = age_pipeline.fit_transform(X_train[['age']])
# print(f"   Age before: min={age_before.min():.1f}, max={age_before.max():.1f}")
# print(f"   Age after (capped): min={age_after.min():.1f}, max={age_after.max():.1f}")

# # Kiểm tra scaling
# print("\n3. Scaling check (on KNN pipeline):")
# X_scaled = preprocess_scaled.fit_transform(X_train)
# # Lấy numerical features đã scaled (vị trí đầu tiên)
# num_scaled = X_scaled[:, :len(num_features)]
# print(f"   Scaled numerical features - mean~0: {num_scaled.mean():.4f}, std~1: {num_scaled.std():.4f}")

# # Kiểm tra encoding
# print("\n4. Encoding check:")
# cat_encoder = cat_pipeline.named_steps['encoder']
# cat_encoder.fit(X_train[cat_features])
# print(f"   Categorical features encoded to {cat_encoder.get_feature_names_out(cat_features).shape[0]} columns")

# print("\n" + "="*60)
# print("✅ QUY TRÌNH HOÀN TẤT")
# print("="*60)